In [1]:
import torch
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 3050

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import DenseNet201_Weights, DenseNet169_Weights
import timm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
from tqdm import tqdm  # Progress bars

In [3]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
data_dir = '/home/rifat-cou/Documents/Project/Dataset_Raw'  # Your folder
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
class_names = full_dataset.classes  # For classification_report

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform

# Loaders
batch_size = 16  # For 6GB VRAM; reduce if needed
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
class_names = full_dataset.classes
print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

cuda
Classes: ['Chikenpox', 'Cowpox', 'Measles', 'MonkeyPox', 'Normal']
Train: 2088, Val: 523


In [5]:
# DenseNet201 (best at 300 epochs)
model1 = models.densenet201(weights=DenseNet201_Weights.DEFAULT)
model1.classifier = nn.Linear(model1.classifier.in_features, num_classes)
for param in model1.features[:8].parameters():
    param.requires_grad = False
model1 = model1.to(device)

# DenseNet169 (second at 100 epochs)
model2 = models.densenet169(weights=DenseNet169_Weights.DEFAULT)
model2.classifier = nn.Linear(model2.classifier.in_features, num_classes)
for param in model2.features[:8].parameters():
    param.requires_grad = False
model2 = model2.to(device)

# EfficientNetB3 (third at 300 epochs)
model3 = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
for param in model3.blocks[:4].parameters():
    param.requires_grad = False
model3 = model3.to(device)

In [8]:
def train_single_model(model, epochs, save_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    best_acc = 0.0
    unfreeze_epoch = epochs // 2  # Unfreeze mid-way

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validate
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(val_labels, val_preds)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {train_loss/len(train_loader):.4f} - Val Acc: {acc:.4f}')
        
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'{save_name}_best.pth')
        
        scheduler.step()

        # Unfreeze more
        if epoch == unfreeze_epoch:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)

# Train each
train_single_model(model1, 300, 'densenet201')
train_single_model(model2, 100, 'densenet169')
train_single_model(model3, 300, 'efficientnetb3')

100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.02it/s]


Epoch 1/300 - Loss: 0.1164 - Val Acc: 0.9790


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.05it/s]


Epoch 2/300 - Loss: 0.0762 - Val Acc: 0.9732


100%|█████████████████████████████████████████| 131/131 [00:32<00:00,  4.05it/s]


...

Epoch 298/300 - Loss: 0.0000 - Val Acc: 0.9847


100%|█████████████████████████████████████████| 131/131 [00:29<00:00,  4.48it/s]


Epoch 299/300 - Loss: 0.0000 - Val Acc: 0.9809


100%|█████████████████████████████████████████| 131/131 [00:29<00:00,  4.46it/s]


Epoch 300/300 - Loss: 0.0000 - Val Acc: 0.9847


In [9]:
# Load best models
model1.load_state_dict(torch.load('densenet201_best.pth'))
model2.load_state_dict(torch.load('densenet169_best.pth'))
model3.load_state_dict(torch.load('efficientnetb3_best.pth'))

models_list = [model1, model2, model3]

def ensemble_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
        return torch.argmax(avg_probs, dim=1)

In [10]:
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = ensemble_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

# Save
model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'V1_simple_avg_top3')
os.makedirs(ensemble_dir, exist_ok=True)
torch.save(model1.state_dict(), os.path.join(ensemble_dir, 'densenet201.pth'))
torch.save(model2.state_dict(), os.path.join(ensemble_dir, 'densenet169.pth'))
torch.save(model3.state_dict(), os.path.join(ensemble_dir, 'efficientnetb3.pth'))

              precision    recall  f1-score   support

   Chikenpox       0.99      1.00      0.99        99
      Cowpox       1.00      1.00      1.00       116
     Measles       1.00      1.00      1.00       102
   MonkeyPox       1.00      0.99      1.00       117
      Normal       1.00      1.00      1.00        89

    accuracy                           1.00       523
   macro avg       1.00      1.00      1.00       523
weighted avg       1.00      1.00      1.00       523

Final Val Accuracy: 0.9981 - F1 Score: 0.9981


In [11]:
# Assuming models_list = [model1, model2, model3] are loaded
def ensemble_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1)

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = ensemble_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

              precision    recall  f1-score   support

   Chikenpox       0.99      1.00      0.99        99
      Cowpox       1.00      1.00      1.00       116
     Measles       1.00      1.00      1.00       102
   MonkeyPox       1.00      0.99      1.00       117
      Normal       1.00      1.00      1.00        89

    accuracy                           1.00       523
   macro avg       1.00      1.00      1.00       523
weighted avg       1.00      1.00      1.00       523

Final Val Accuracy: 0.9981 - F1 Score: 0.9981


In [12]:
class SimpleAvgEnsembleONNX(nn.Module):
    def __init__(self, base_models):
        super(SimpleAvgEnsembleONNX, self).__init__()
        self.bases = nn.ModuleList(base_models)

    def forward(self, x):
        probs = [torch.softmax(base(x), dim=1) for base in self.bases]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
        return avg_probs  # Return probs for flexibility (argmax in post-processing)

# Create combined model
ensemble_onnx_model = SimpleAvgEnsembleONNX(models_list)
ensemble_onnx_model.eval()

SimpleAvgEnsembleONNX(
  (bases): ModuleList(
    (0): DenseNet(
      (features): Sequential(
        (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu0): ReLU(inplace=True)
        (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (denseblock1): _DenseBlock(
          (denselayer1): _DenseLayer(
            (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu1): ReLU(inplace=True)
            (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu2): ReLU(inplace=True)
            (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          )
          (denselayer2): _DenseLa

In [13]:
# Move to CPU for export
ensemble_onnx_model.to('cpu')
for base in ensemble_onnx_model.bases:
    base.to('cpu')

dummy_input = torch.randn(1, 3, 224, 224)  # Batch=1

onnx_path = os.path.join(ensemble_dir, 'simple_avg_ensemble_v1.onnx')
torch.onnx.export(
    ensemble_onnx_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f'ONNX exported: {onnx_path}')

# Optional verification
import onnxruntime as ort
ort_session = ort.InferenceSession(onnx_path)
ort_input = {ort_session.get_inputs()[0].name: dummy_input.numpy()}
ort_out = ort_session.run(None, ort_input)
print('ONNX verification passed')

ONNX exported: /home/rifat-cou/Documents/Project/Ensemble Models/V1_simple_avg_top3/simple_avg_ensemble_v1.onnx
ONNX verification passed
